# Sentiment Analysis IKN — Fine-tuning IndoBERT

Notebook ini melanjutkan dari `eda_preprocessing.ipynb`.

**Model yang digunakan:** `indobenchmark/indobert-base-p1` — BERT yang di-pretrain pada korpus Bahasa Indonesia.

**Pipeline:**
1. Install & import library
2. Load & encode data
3. Split dataset
4. Tokenisasi dengan IndoBERT Tokenizer
5. Buat Dataset & DataLoader
6. Definisi model
7. Training loop
8. Evaluasi (accuracy, classification report, confusion matrix)
9. Simpan model
10. Inference / prediksi teks baru

## 1. Install Library
> Jalankan cell ini sekali saja. Jika sudah terinstall, bisa di-skip.

In [ ]:
!pip install transformers datasets torch scikit-learn seaborn matplotlib --quiet

## 2. Import Library

In [ ]:
import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns

from torch import nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import (
    BertTokenizer,
    BertForSequenceClassification,
    get_linear_schedule_with_warmup
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)
from sklearn.preprocessing import LabelEncoder

# Cek device: GPU atau CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device yang digunakan: {device}')

In [ ]:
df = pd.read_csv('../data/clean_data.csv')

# Hapus kolom bantu jika ada
if 'text_length' in df.columns:
    df = df.drop(columns=['text_length'])

df = df.dropna(subset=['text', 'label'])
df = df.reset_index(drop=True)

print(f'Total data: {len(df)}')
print(df['label'].value_counts())
df.head()

In [ ]:
# Encode label: negatif=0, positif=1
le = LabelEncoder()
df['label_enc'] = le.fit_transform(df['label'])

print('Mapping label:')
for i, cls in enumerate(le.classes_):
    print(f'  {cls} -> {i}')

In [ ]:
X = df['text'].values
y = df['label_enc'].values

# 70% train | 15% val | 15% test
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print(f'Train : {len(X_train)} samples')
print(f'Val   : {len(X_val)} samples')
print(f'Test  : {len(X_test)} samples')